# Practice 1.2 - Vietnamese Word Segmentation

Objectives:
- Implement Vietnamese word segmentation with **forward maximum matching**.
- Normalize Unicode consistently before dictionary lookup.
- Evaluate predictions with **precision / recall / F1** on word boundaries.


## Table of Contents

### Section 1: Setup
1. Resolve data paths for the dictionary and evaluation files.
2. Inspect a few raw sentences from the corpus-derived evaluation set.

### Section 2: Exercise
1. Reuse the provided normalization and tokenization helpers.
2. Implement `load_dictionary(...)`.
3. Implement `segment_text(...)` with forward maximum matching.
4. Implement `evaluate(...)` for token-boundary metrics.

### Section 3: Playground
1. Segment a single sentence.
2. Run the model on `eval_input.txt`.
3. Compare predictions with `eval_gold.txt` after finishing the TODOs.


## Section 1: Setup

The notebook works whether you open it from the repository root or from inside the `Practice/1-2 Word Segmentation` folder.


In [ ]:
from pathlib import Path
import unicodedata
from typing import Iterable

def resolve_base_dir() -> Path:
    markers = (
        Path('vn-dict.txt'),
        Path('data/eval_input.txt'),
    )
    cwd = Path.cwd().resolve()

    for search_root in (cwd, *cwd.parents):
        for candidate in (search_root, search_root / 'Practice/1-2 Word Segmentation'):
            if all((candidate / marker).exists() for marker in markers):
                return candidate

    raise FileNotFoundError(
        'Could not locate Practice/1-2 Word Segmentation from the current working directory.'
    )

base_dir = resolve_base_dir()

dict_path = base_dir / 'vn-dict.txt'
eval_input_path = base_dir / 'data' / 'eval_input.txt'
eval_gold_path = base_dir / 'data' / 'eval_gold.txt'

print(f'Base directory: {base_dir.resolve()}')
print(f'Dictionary exists: {dict_path.exists()}')
print(f'Eval input exists: {eval_input_path.exists()}')
print(f'Eval gold exists: {eval_gold_path.exists()}')


In [ ]:
sample_sentences = eval_input_path.read_text(encoding='utf-8').splitlines()[:5]
for index, sentence in enumerate(sample_sentences, start=1):
    print(f'{index:02d}. {sentence}')


## Section 2: Exercise

The helper functions below are already complete. Your main work is in `load_dictionary`, `segment_text`, and `evaluate`.

### Guidance for `segment_text`
- Start from `tokens = tokenize_text(text)` so punctuation has already been split into separate tokens.
- Sweep from left to right with an index over the token list.
- At each position, try the longest possible window first: `min(max_word_len, remaining_tokens)` down to `1`.
- Build each lookup candidate with spaces between syllables, then normalize before checking membership in `lexicon`.
- If a window matches, keep the original token surface form and join that span with `_` in the output.
- If nothing matches, emit the current token unchanged and move forward by one token.

### Guidance for `evaluate`
- Convert each segmented sentence into two things: the underlying syllable sequence and the set of word-boundary spans.
- Example: `hoc_sinh gioi` corresponds to syllables `['hoc', 'sinh', 'gioi']` and spans `{(0, 2), (2, 3)}`.
- Do this for both prediction and gold after removing `_` and normalizing whitespace / Unicode.
- Check that prediction and gold have the same syllable sequence before scoring; otherwise raise a `ValueError`.
- Count `matched = pred_boundaries & gold_boundaries`, then accumulate totals across all sentences.
- Compute precision, recall, and F1 with zero-division guards.


In [ ]:
EDGE_PUNCTUATION = set("\"'“”‘’.,!?;:()[]{}|-")

def normalize_unicode(text: str) -> str:
    return unicodedata.normalize('NFC', text)

def normalize_whitespace(text: str) -> str:
    return ' '.join(normalize_unicode(text).split())

def tokenize_text(text: str) -> list[str]:
    tokens: list[str] = []
    for raw_token in normalize_whitespace(text).split():
        leading: list[str] = []
        trailing: list[str] = []
        core = raw_token

        while core and core[0] in EDGE_PUNCTUATION:
            leading.append(core[0])
            core = core[1:]

        while core and core[-1] in EDGE_PUNCTUATION:
            trailing.append(core[-1])
            core = core[:-1]

        tokens.extend(leading)
        if core:
            tokens.append(core)
        tokens.extend(reversed(trailing))

    return tokens


In [ ]:
def load_dictionary(path: str | Path) -> tuple[set[str], int]:
    # TODO: Read vn-dict.txt, normalize Unicode / whitespace,
    # remove duplicates, and return (lexicon, max_word_len).
    raise NotImplementedError


In [ ]:
def segment_text(text: str, lexicon: set[str], max_word_len: int) -> list[str]:
    # TODO: Implement forward maximum matching on syllable-tokenized input.
    # Suggested flow:
    # 1. Tokenize with tokenize_text(text).
    # 2. At each index, try span lengths from longest to shortest.
    # 3. Join the candidate span with spaces for dictionary lookup.
    # 4. If matched, append the original surface tokens joined by '_'.
    # 5. If nothing matches, fall back to the current single token.
    raise NotImplementedError


In [ ]:
def evaluate(pred_sentences: Iterable[str], gold_sentences: Iterable[str]) -> tuple[float, float, float]:
    # TODO: Compute precision, recall, and F1 on word boundaries.
    # Suggested flow:
    # 1. Convert each segmented sentence into syllables + boundary spans.
    # 2. Validate that pred/gold become the same syllable sequence after removing '_'.
    # 3. Count matched spans via set intersection.
    # 4. Accumulate predicted_total and gold_total across all lines.
    # 5. Compute precision, recall, and F1 with zero-division guards.
    raise NotImplementedError


In [ ]:
def read_lines(path: str | Path) -> list[str]:
    return Path(path).read_text(encoding='utf-8').splitlines()

def write_lines(path: str | Path, lines: Iterable[str]) -> None:
    output_path = Path(path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text('\n'.join(lines) + '\n', encoding='utf-8')


## Section 3: Playground

Run the following cells after you finish the TODOs above.


In [ ]:
# Segment one sentence after completing the exercise.
# lexicon, max_word_len = load_dictionary(dict_path)
# print(' '.join(segment_text('Khẩu súng gây án đã bị thu giữ .', lexicon, max_word_len)))


In [ ]:
# Evaluate on the 100-sentence corpus-derived benchmark after completing the exercise.
# lexicon, max_word_len = load_dictionary(dict_path)
# predictions = [' '.join(segment_text(line, lexicon, max_word_len)) for line in read_lines(eval_input_path)]
# precision, recall, f1 = evaluate(predictions, read_lines(eval_gold_path))
# print(f'Precision: {precision:.4f}')
# print(f'Recall: {recall:.4f}')
# print(f'F1: {f1:.4f}')
